### Clone and install the repository

In [1]:
!git clone https://github.com/OpenRLHF/OpenRLHF.git

Cloning into 'OpenRLHF'...
remote: Enumerating objects: 15106, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 15106 (delta 20), reused 19 (delta 17), pack-reused 15068 (from 2)
Receiving objects: 100% (15106/15106), 4.20 MiB | 7.50 MiB/s, done.
Resolving deltas: 100% (11178/11178), done.


In [2]:
%cd OpenRLHF

/home/thuandn/Repository/aio-chatbot-llm-rlhf/notebooks/OpenRLHF


In [ ]:
!pip install openrlhf[vllm_latest]

### Train Reward Model

In [8]:
!CUDA_VISIBLE_DEVICES=0 deepspeed --module openrlhf.cli.train_rm \
   --ckpt.output_dir ./checkpoint/Llama-3.2-1B-rm-dpo \
   --ckpt.save_steps -1 \
   --logger.logging_steps 1 \
   --eval.steps -1 \
   \
   --train.batch_size 512 \
   --train.micro_batch_size 64 \
   --train.max_epochs 1 \
   \
   --model.model_name_or_path thuanan/Llama-3.2-1B-Instruct-Chat-sft \
   --model.gradient_checkpointing_enable \
   \
   --ds.value_head_prefix score \
   --ds.param_dtype bf16 \
   --ds.zero_stage 2 \
   --ds.adam_offload \
   --ds.packing_samples \
   \
   --ds.lora.rank 16 \
   --ds.lora.alpha 32 \
   \
   --data.dataset thuanan/Vi-Alpaca-Preference \
   --data.max_len 2048 \
   --data.apply_chat_template \
   --data.chosen_key chosen \
   --data.rejected_key rejected \
   \
   --adam.lr 5e-6

[2026-04-22 10:48:01,681] [WARNING] [runner.py:232:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
Detected VISIBLE_DEVICES=1: setting --include=localhost:1
[2026-04-22 10:48:01,681] [INFO] [runner.py:630:main] cmd = /home/thuandn/.conda/envs/flashattn_cu130_cp312/bin/python3.12 -u -m deepspeed.launcher.launch --world_info=eyJsb2NhbGhvc3QiOiBbMV19 --master_addr=127.0.0.1 --master_port=29500 --module --enable_each_rank_log=None --log_level=info openrlhf.cli.train_rm --ckpt.output_dir ./checkpoint/Llama-3.2-1B-rm-dpo --ckpt.save_steps -1 --logger.logging_steps 1 --eval.steps -1 --train.batch_size 512 --train.micro_batch_size 64 --train.max_epochs 1 --model.model_name_or_path thuanan/Llama-3.2-1B-Instruct-Chat-sft --model.gradient_checkpointing_enable --ds.value_head_prefix score --ds.param_dtype bf16 --ds.zero_stage 2 --ds.adam_offload --ds.packing_samples --ds.lora.rank 16 --ds.lora.alpha 32 --data.dataset thuanan/Vi-Alpaca-Preference --dat

### Merge LoRA Adapter Weights

In [1]:
!python -m openrlhf.cli.lora_combiner \
   --model_path /mnt/dataset1/thuannd/Repository/aivn-aio2024-chatbot-llm-rlhf/checkpoint/Llama-3.2-1B-rm-init \
   --lora_path ./checkpoint/Llama-3.2-1B-rm-dpo \
   --output_path ./checkpoint/Llama-3.2-1B-rm-dpo-combined \
   --is_rm \
   --bf16

usage: lora_combiner.py [-h] --model_path MODEL_PATH --lora_path LORA_PATH
                        --output_path OUTPUT_PATH [--is_rm]
                        [--ds.param_dtype {bf16,fp16}]
lora_combiner.py: error: unrecognized arguments: --bf16


### Push to Hugging Face Hub

In [12]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ckpt_path = "./checkpoint/Llama-3.2-1B-rm-dpo-combined"
tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
model = AutoModelForSequenceClassification.from_pretrained(ckpt_path)

In [13]:
type(model)

transformers.models.llama.modeling_llama.LlamaForSequenceClassification

In [14]:
model

LlamaForSequenceClassification(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((20

In [31]:
model.push_to_hub(
    "thuanan/Llama-3.2-1B-RM-DPO",
    commit_message="Add model ckpt",
)

model.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/thuanan/Llama-3.2-1B-RM-DPO/commit/35273dea7e8a6fa128ed8d53a7e4b429ad12ffaa', commit_message='Add model ckpt', commit_description='', oid='35273dea7e8a6fa128ed8d53a7e4b429ad12ffaa', pr_url=None, repo_url=RepoUrl('https://huggingface.co/thuanan/Llama-3.2-1B-RM-DPO', endpoint='https://huggingface.co', repo_type='model', repo_id='thuanan/Llama-3.2-1B-RM-DPO'), pr_revision=None, pr_num=None)

In [32]:
tokenizer.push_to_hub(
    "thuanan/Llama-3.2-1B-RM-DPO",
    commit_message="Add tokenizer",
)

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/thuanan/Llama-3.2-1B-RM-DPO/commit/740231d7bb2e843aadeea234cb5088f5b8a43140', commit_message='Add tokenizer', commit_description='', oid='740231d7bb2e843aadeea234cb5088f5b8a43140', pr_url=None, repo_url=RepoUrl('https://huggingface.co/thuanan/Llama-3.2-1B-RM-DPO', endpoint='https://huggingface.co', repo_type='model', repo_id='thuanan/Llama-3.2-1B-RM-DPO'), pr_revision=None, pr_num=None)

### Test Reward Model

In [26]:
inputs = tokenizer(
    "Tại sao bạn lại thích học lập trình?",
    return_tensors="pt",
    max_length=2048,
    truncation=True,
)

In [27]:
model

LlamaForSequenceClassification(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((20

In [28]:
reward = model.model(**inputs).last_hidden_state
reward


tensor([[[ 2.3110,  3.2832,  0.0715,  ..., -0.8754, -2.5920,  1.2897],
         [ 0.1436, -0.1955,  0.9064,  ..., -1.4842, -3.2248, -0.3736],
         [ 0.1863,  3.4998,  1.6896,  ...,  1.3823, -2.3932,  1.7603],
         ...,
         [-1.9774,  4.3347, -0.2467,  ...,  1.8927,  1.4284,  2.0450],
         [-1.4019,  2.4985,  1.7585,  ..., -1.9504, -1.1227,  2.3001],
         [-1.0206,  2.1333,  2.1011,  ..., -1.6172, -1.9382, -1.5404]]],
       grad_fn=<MulBackward0>)

In [29]:
reward = model.score(reward)[:, -1]
reward

tensor([[0.0406]], grad_fn=<SelectBackward0>)